# 02 — Data extraction

Read all the simulation ROOT files (~1000 files at ~25 MB each = ~21 GB) and reduce them to a single small (~350 KB) `.npz` file containing per-event aggregate features for the ML model:

- `all_truth`: true photon energy per event
- `all_visible`: total visible energy (analog readout)
- `all_mip`: MIP-equivalent hit count (each hit weighted by E_hit / E_MIP)
- `all_hits`: raw pixel-hit count (digital readout)
- `all_cluster`: largest 2D cluster size on a single layer (naive clustering readout)

This is the heavy step in the pipeline: it forks a process pool to read files in parallel and takes 5-20 minutes depending on EAF load.

**Kernel**: `Python (Key4hep)` (CPU is fine, this is I/O-bound).

**Output**: `$CALOMAPS_HOME/models/decal_extracted_data.npz`.


## 1. Setup

In [ ]:
import os, glob, uproot, awkward as ak, numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed

CALOMAPS_HOME = os.environ.get("CALOMAPS_HOME", os.path.expanduser("~/CALOMAPS"))
DATA_BASE = os.environ.get("CALOMAPS_DATA_BASE", os.path.expanduser("~/CALOMAPS-data"))
DATASET = "data_spectrum_100um_400GeV"

# Cell size: matches geometry/SiD_TestBeam.xml. Used by the cluster-size readout.
CELL_SIZE = 0.1   # mm — 100 micron pixels

# MIP energy = most probable energy deposit of a minimum-ionizing particle in 320 um Si
E_MIP = 78e-6     # GeV (~78 keV; rough average MIP for the geometry we have)

# Bins for the optional shower-profile aggregation (not strictly needed downstream)
bins = [(5, 20), (20, 50), (50, 100), (100, 200), (200, 400)]
num_layers = 30


## 2. Per-file worker function

In [ ]:
def process_single_file(filepath):
    """Extract per-event features from one ROOT file. Returns a tuple of lists."""
    try:
        with uproot.open(filepath) as f:
            tree = f["events"]
            x_j = tree["ECalBarrelHits/ECalBarrelHits.position.x"].array()
            y_j = tree["ECalBarrelHits/ECalBarrelHits.position.y"].array()
            z_j = tree["ECalBarrelHits/ECalBarrelHits.position.z"].array()
            e_j = tree["ECalBarrelHits/ECalBarrelHits.energy"].array()
            p_j = tree["MCParticles/MCParticles.momentum.y"].array()
    except Exception as e:
        print(f"  failed: {os.path.basename(filepath)}: {e}")
        return None

    truths, visibles, mips, hits, clusters = [], [], [], [], []
    p_sum = {b: np.zeros(num_layers) for b in range(len(bins))}
    p_counts = {b: 0 for b in range(len(bins))}

    for i in range(len(x_j)):
        x = x_j[i].to_numpy()
        y = y_j[i].to_numpy()
        z = z_j[i].to_numpy()
        e = e_j[i].to_numpy()
        if len(p_j[i]) == 0:
            continue
        E_true = float(p_j[i][0])

        # Filter to +Y face only (where the photon enters)
        mask = (np.abs(x) <= 50) & (np.abs(z) <= 50) & (y >= 1260) & (y <= 1420)
        x, y, z, e = x[mask], y[mask], z[mask], e[mask]
        if len(x) == 0:
            continue

        E_vis = float(e.sum())
        n_hits = int(len(x))
        n_mip = float((e / E_MIP).sum())

        # Naive cluster size: pick the densest layer, then the largest contiguous
        # blob of cells (using grid bucketing for speed).
        unique_y, counts = np.unique(np.round(y, 2), return_counts=True)
        if len(unique_y) == 0:
            cluster_size = 0
        else:
            densest_y = unique_y[np.argmax(counts)]
            in_layer = np.abs(y - densest_y) < 0.05
            xi = np.round(x[in_layer] / CELL_SIZE).astype(int)
            zi = np.round(z[in_layer] / CELL_SIZE).astype(int)
            if len(xi) == 0:
                cluster_size = 0
            else:
                # Connected components via BFS over the set of lit cells
                lit = set(zip(xi, zi))
                seen = set()
                best = 0
                for c in lit:
                    if c in seen:
                        continue
                    queue, comp = [c], 0
                    while queue:
                        u = queue.pop()
                        if u in seen:
                            continue
                        seen.add(u)
                        comp += 1
                        for du in [(-1,0),(1,0),(0,-1),(0,1)]:
                            v = (u[0]+du[0], u[1]+du[1])
                            if v in lit and v not in seen:
                                queue.append(v)
                    if comp > best:
                        best = comp
                cluster_size = best

        truths.append(E_true)
        visibles.append(E_vis)
        mips.append(n_mip)
        hits.append(n_hits)
        clusters.append(cluster_size)

    return p_sum, p_counts, truths, visibles, mips, hits, clusters


## 3. Run the extractor in parallel

In [ ]:
file_list = sorted(glob.glob(os.path.join(DATA_BASE, DATASET, "sim_photons_part*.root")))
print(f"Found {len(file_list)} files. Starting parallel extraction...")

all_truth, all_visible, all_mip, all_hits, all_cluster = [], [], [], [], []

# Drop max_workers to ~8 if you see memory pressure; 64 is aggressive on a small node
with ProcessPoolExecutor(max_workers=16) as executor:
    futures = {executor.submit(process_single_file, f): f for f in file_list}
    for count, fut in enumerate(as_completed(futures), 1):
        result = fut.result()
        if result:
            _, _, t, v, m, h, c = result
            all_truth.extend(t); all_visible.extend(v); all_mip.extend(m)
            all_hits.extend(h); all_cluster.extend(c)
        if count % 50 == 0:
            print(f"  processed {count}/{len(file_list)} files...")

all_truth = np.array(all_truth)
all_visible = np.array(all_visible, dtype=np.float32)
all_mip = np.array(all_mip, dtype=np.float32)
all_hits = np.array(all_hits, dtype=np.int64)
all_cluster = np.array(all_cluster, dtype=np.int64)

print(f"\nExtraction complete. {len(all_truth)} events extracted.")


## 4. Save the .npz

In [ ]:
out_dir = os.path.join(CALOMAPS_HOME, "models")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "decal_extracted_data.npz")
np.savez_compressed(out_path,
                    all_truth=all_truth, all_visible=all_visible,
                    all_mip=all_mip, all_hits=all_hits, all_cluster=all_cluster)
print(f"Saved {out_path} ({os.path.getsize(out_path)/1024:.0f} KB)")


Done. The extracted features are now ready for the ML pipeline — see [`03_ml_training_and_eval.ipynb`](03_ml_training_and_eval.ipynb).
